# Performance Tuning: Partitions, Shuffling, and Joins

Welcome to the fourth notebook in our series! Now that you know how to write transformations and query data using SQL, it is time to look under the hood at **Cluster Performance**. 

When scaling up from local files to terabytes of cluster data, poorly written code can cause network bottlenecks, out-of-memory errors, and severe performance degradation.

---

## 1. Understanding Partitions and Parallelism

Spark’s speed comes from processing data in parallel across multiple nodes. The fundamental unit of parallelism in Spark is the **Partition**—a logical chunk of data stored on a single node.

* **Too Few Partitions:** Your cluster's executors will sit idle while a single node struggles to process a massive chunk of data.
* **Too Many Partitions:** Spark will spend more time managing metadata and task scheduling overhead than actually processing data.
* **Controlling Partitions:**
  * `repartition(n)`: Full shuffle. Increases or decreases partitions evenly. Use when scaling up parallelism.
  * `coalesce(n)`: No-shuffle optimization. Only *decreases* partitions by combining existing ones locally. Much faster than repartitioning when reducing size.

## 2. The Cost of Shuffling

As covered in earlier notes, **Wide Transformations** (like `groupBy`, `join`, or `distinct`) require a **shuffle**.

* A shuffle is the process where data is resorted and sent across the cluster network so that all rows with the same key end up on the same executor.
* Network I/O is vastly slower than memory or CPU processing. Minimizing shuffles is the #1 rule of Spark performance tuning.

## 3. Optimizing Joins: Broadcast Hash Joins

When joining a **large table** with a **small table** (e.g., transactions joined with a lookup category table), a standard shuffle join forces both tables to shuffle across the network.

* **Broadcast Join Solution:** Instead of shuffling both tables, Spark can copy (broadcast) the entire small table to every single executor node.
* Each node performs the join locally in memory with **zero network shuffle**, dramatically accelerating execution speed.

## 4. Setting up the Environment

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import broadcast

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("PerformanceTuning") \
    .getOrCreate()

# Create sample large dataset (Transactions)
tx_data = [(1, 101, 5), (2, 102, 2), (3, 101, 1), (4, 103, 10), (5, 102, 3)]
tx_df = spark.createDataFrame(tx_data, ["TxID", "ProductID", "Quantity"])

# Create sample small dataset (Lookup Table)
prod_data = [(101, "Laptop"), (102, "Mouse"), (103, "Keyboard")]
prod_df = spark.createDataFrame(prod_data, ["ProductID", "ProductName"])

print(f"Initial Transaction Partitions: {tx_df.rdd.getNumPartitions()}")

## 5. Demonstrating Partition Management (`repartition` vs `coalesce`)

In [ ]:
# Increase partitions using repartition (triggers a shuffle)
repartitioned_df = tx_df.repartition(4)
print(f"Partitions after repartition(4): {repartitioned_df.rdd.getNumPartitions()}")

# Decrease partitions using coalesce (avoids a full shuffle)
coalesced_df = repartitioned_df.coalesce(2)
print(f"Partitions after coalesce(2): {coalesced_df.rdd.getNumPartitions()}")

## 6. Implementing a Broadcast Join

Let's explicitly tell Spark to broadcast the smaller product table to avoid an expensive cluster-wide shuffle during the join operation.

In [ ]:
# Use the broadcast hint to optimize the join
optimized_join_df = tx_df.join(
    broadcast(prod_df), 
    tx_df["ProductID"] == prod_df["ProductID"]
)

# Action: Show the result
optimized_join_df.show()

## Summary Checklist for Performance

* **Monitor Shuffles:** Always check the Spark UI for shuffle read/write sizes.
* **Use Broadcasts:** Broadcast small lookup tables whenever joining with large fact tables.
* **Coalesce Wisely:** Use `coalesce()` instead of `repartition()` when reducing file partition counts before writing output data.